In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Dark Theme ────────────────────────────────────────────────────────────────
PAPER_BG   = '#0D1117'
PLOT_BG    = '#161B22'
GRID_C     = '#30363D'
FONT_C     = '#E6EDF3'
MUTED_C    = '#8B949E'
TEMPLATE   = 'plotly_dark'

CITY_COLORS = {
    'Bangkok':      '#FF6B6B',
    'Singapore':    '#74B9FF',
    'Kuala Lumpur': '#4ECDC4',
    'Jakarta':      '#FFB347',
    'Manila':       '#C77DFF',
}

def base_layout(height=460, legend_h=True):
    leg = dict(bgcolor='rgba(22,27,34,0.9)', bordercolor=GRID_C, borderwidth=1, font_size=11)
    if legend_h:
        leg.update(orientation='h', y=1.13, x=1, xanchor='right')
    return dict(
        template=TEMPLATE, paper_bgcolor=PAPER_BG, plot_bgcolor=PLOT_BG,
        font=dict(color=FONT_C, family='Segoe UI, Arial, sans-serif'),
        height=height, legend=leg,
        margin=dict(l=60, r=40, t=75, b=55),
        hoverlabel=dict(bgcolor='#1F2937', bordercolor=GRID_C, font_size=12),
    )

def ax_style():
    return dict(gridcolor=GRID_C, zerolinecolor=GRID_C)

print('✅ Setup complete — Dark theme loaded')

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
df = pd.read_csv('ASEAN_Urban_Growth_Final_with_Mode.csv')
df['Date']  = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# ── Mode label maps ───────────────────────────────────────────────────────────
MODE_BKK = {'BTS':'BTS Skytrain','MRT':'MRT Blue/Purple','SRT':'SRT Red Line',
             'ARL':'Airport Rail Link','YL':'MRT Yellow Line','PK':'MRT Pink Line','RL':'Regional Rail'}
MODE_SGP = {'MRT':'MRT','Public Bus':'Public Bus','LRT':'LRT'}
MODE_KL  = {
    'rail_lrt_kj':'LRT Kelana Jaya','rail_mrt_kajang':'MRT Kajang',
    'rail_lrt_ampang':'LRT Ampang','bus_rkl':'RapidKL Bus',
    'rail_mrt_pjy':'MRT Putrajaya','rail_monorail':'KL Monorail',
    'rail_komuter':'KTM Komuter','rail_komuter_utara':'KTM Utara',
    'rail_ets':'ETS Train','rail_intercity':'Intercity Rail',
    'rail_tebrau':'KTM Tebrau','bus_rkn':'RapidKN Bus','bus_rpn':'RapidPN Bus'
}
MODE_JKT = {'TRANSJAKARTA':'TransJakarta (BRT)','KRL':'KRL Commuter',
             'MRT':'MRT Jakarta','LRT':'LRT Jakarta',
             'BUS SEKOLAH':'School Bus','KCI COMMUTER BANDARA':'Airport Rail','KAPAL':'Ferry'}

print(f'Dataset: {df.shape[0]:,} rows | {df["City"].nunique()} cities')
print('Cities:', df['City'].unique().tolist())

In [ ]:
# ── Singapore: prep data ──────────────────────────────────────────────────────
sgp = df[(df['City']=='Singapore') & (df['Year'].between(2019,2024))].copy()
sgp['Mode_Label'] = sgp['Mode'].map(lambda x: MODE_SGP.get(x,x))

sgp_m  = sgp.groupby('Date')['Ridership'].sum().reset_index()
sgp_yr = sgp.groupby(['Year','Mode_Label'])['Ridership'].sum().reset_index()
sgp_sh = sgp.groupby('Mode_Label')['Ridership'].sum().reset_index().sort_values('Ridership',ascending=False)

yoy_list=[]
for mode in sgp['Mode_Label'].unique():
    sub = sgp_yr[sgp_yr['Mode_Label']==mode].sort_values('Year').copy()
    sub['YoY'] = sub['Ridership'].pct_change()*100
    yoy_list.append(sub)
sgp_yoy = pd.concat(yoy_list).dropna(subset=['YoY'])

print('Singapore data ready ✅')

In [ ]:
fig = px.area(
    sgp_m, x='Date', y='Ridership',
    title='<b>ตารางที่ 2.1</b>  สิงคโปร์ — ผู้โดยสารรายเดือน ทุกระบบรวม (2019–2024)',
    labels={'Ridership':'จำนวนผู้โดยสาร','Date':''},
    color_discrete_sequence=[CITY_COLORS['Singapore']],
)
fig.update_traces(fillcolor='rgba(116,185,255,0.13)', line_color=CITY_COLORS['Singapore'], line_width=2.2)
fig.update_layout(**base_layout(420), hovermode='x unified',
                  yaxis=dict(tickformat='.2s',**ax_style()), xaxis=ax_style())
fig.show()

In [ ]:
fig = px.pie(
    sgp_sh, names='Mode_Label', values='Ridership',
    title='<b>ตารางที่ 2.2</b>  สิงคโปร์ — สัดส่วนผู้โดยสารแต่ละระบบ (2019–2024 รวม)',
    hole=0.45, color_discrete_sequence=['#74B9FF','#A8E6CF','#DDA0DD'],
)
fig.update_traces(textposition='inside', textinfo='percent+label',
                  insidetextorientation='radial', textfont_size=11,
                  marker=dict(line=dict(color=PAPER_BG,width=2)))
fig.update_layout(**base_layout(440, legend_h=False),
                  legend=dict(bgcolor='rgba(22,27,34,0.9)',bordercolor=GRID_C,
                              borderwidth=1,font_size=11,orientation='h',y=-0.1))
fig.show()

In [ ]:
fig = px.bar(
    sgp_yr, x='Year', y='Ridership', color='Mode_Label',
    barmode='stack',
    title='<b>ตารางที่ 2.3</b>  สิงคโปร์ — ผู้โดยสารแต่ละระบบขนส่ง รายปี (2019–2024)',
    labels={'Ridership':'ผู้โดยสาร','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=['#74B9FF','#A8E6CF','#DDA0DD'],
)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(tickformat='.2s',**ax_style()), bargap=0.25)
fig.show()

In [ ]:
fig = px.bar(
    sgp_yoy, x='Year', y='YoY', color='Mode_Label',
    barmode='group', text_auto='.0f',
    title='<b>ตารางที่ 2.4</b>  สิงคโปร์ — อัตราการเปลี่ยนแปลงผู้โดยสาร YoY (%) แต่ละระบบ',
    labels={'YoY':'YoY Change (%)','Mode_Label':'ระบบขนส่ง','Year':''},
    color_discrete_sequence=['#74B9FF','#A8E6CF','#DDA0DD'],
)
fig.add_hline(y=0, line_dash='dash', line_color=MUTED_C, line_width=1.2)
fig.update_layout(**base_layout(460),
                  xaxis=dict(dtick=1,**ax_style()),
                  yaxis=dict(ticksuffix='%',**ax_style()))
fig.update_traces(textfont_size=9, textposition='outside')
fig.show()